In [1]:
X = [
       "This is very Excellent.",
       "This is very good.",
       "This is very okish , normal, fine.",
       "This is very bad, not so good.",
       "This is worst.",
       ]
y = [0,1,2,3,4]

In [2]:
from collections import Counter
from nltk import PorterStemmer
import numpy as np
from scipy.sparse import csr_matrix
import re

In [3]:
sample1 = re.sub(r"[^\w\s]","",X[0])
sample1

'This is very Excellent'

In [4]:
stemmer = PorterStemmer()
sample1=stemmer.stem(sample1)
sample1

'this is very excel'

In [5]:
counter_sample1 = Counter(sample1.split())
counter_sample1,counter_sample1.most_common()

(Counter({'this': 1, 'is': 1, 'very': 1, 'excel': 1}),
 [('this', 1), ('is', 1), ('very', 1), ('excel', 1)])

In [19]:
from sklearn.base import TransformerMixin,BaseEstimator
class Vectorizer(BaseEstimator,TransformerMixin):
    def __init__(self,vocab_size=50):
        self.vocab_size = vocab_size
    def fit(self,X,y=None):
        PrevGodCounter = Counter()
        GodCounter = Counter()
        counters = self.process(X)
        for counter in counters:
            for word,count in counter.items():
                PrevGodCounter[word]+=count
        for word,count in PrevGodCounter.most_common()[:self.vocab_size]:
            GodCounter[word] = count
        self.vocab = {word:i+1 for i,(word,count) in enumerate(GodCounter.most_common()[:self.vocab_size])}
        return self
    def transform(self,X,y=None):
        counters = self.process(X)
        rows=[]
        cols=[]
        data=[]
        for row,counter in enumerate(counters):
            for word,count in counter.items():
                rows.append(row)
                cols.append(self.vocab.get(word,0))
                data.append(count)
        return csr_matrix((data,(rows,cols)),shape=(len(counters),self.vocab_size+1))
    def process(self,X):
        counters = []
        texts = []
        for i in X:
            texts.append(stemmer.stem(re.sub(r"[^\w\s]","",i)))
        for i in texts:
            counters.append(Counter(i.split()))
        return counters

In [20]:
vectorizer = Vectorizer()
a=vectorizer.fit_transform(X)

In [21]:
a.toarray()

array([[0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]], dtype=int32)

In [22]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(a,y)

LogisticRegression()

In [23]:
model.predict(vectorizer.transform(["This is so bad."]))

array([3])

In [24]:
from joblib import load,dump
dump(vectorizer,"vectorizer.joblib")
dump(model,"model.joblib")

['model.joblib']